In [ ]:

import requests
from bs4 import BeautifulSoup
import pandas as pd

# Cabecera estándar para identificarnos ante el servidor
headers = {"User-Agent": "Proyecto de Lenguaje de Programacion"}

lista_noticias = []
# Recorremos las páginas 1 y 2 para recolectar una cantidad moderada de datos
paginas = [1, 2]

for pagina in paginas:
    URL_MENEAME = f"https://www.meneame.net/?page={pagina}"
    print(f"Extrayendo datos de: {URL_MENEAME}")
    
    response = requests.get(URL_MENEAME, headers=headers, timeout=10)
    
    if response.status_code == 200:
        soup = BeautifulSoup(response.text, "html.parser")
        noticias_html = soup.select("div.news-summary")
        
        for noticia in noticias_html:
            bloque_titulo = noticia.select_one("div.center-content h2 a")
            if not bloque_titulo: 
                continue
                
            titulo = bloque_titulo.get_text().strip()
            enlace = bloque_titulo["href"]
            
            # Extraer votos y clics (si no existen, asignamos "0")
            bloque_votos = noticia.select_one("div.votes a")
            votos = bloque_votos.get_text().strip() if bloque_votos else "0"
            
            bloque_clics = noticia.select_one("div.clics")
            clics = bloque_clics.get_text().strip() if bloque_clics else "0 clics"
            
            lista_noticias.append({
                "Título": titulo,
                "Enlace": enlace,
                "Votos_Texto": votos,
                "Clics_Texto": clics
            })
            


# ANÁLISIS ESTADÍSTICO DE FRAUDE / BULOS CON PANDAS

df_noticias = pd.DataFrame(lista_noticias)

# Limpieza: Convertir los textos a números puros 
df_noticias["Votos"] = df_noticias["Votos_Texto"].str.replace(r"\D", "", regex=True).astype(int)
df_noticias["Clics"] = df_noticias["Clics_Texto"].str.replace(r"\D", "", regex=True).astype(int)

#  Calcular el porcentaje de validación social
# 
df_noticias["Ratio_Validacion_%"] = (df_noticias["Votos"] / (df_noticias["Clics"] + 1)) * 100

# Si el ratio es menor al 3%, significa que la noticia 
# genera mucha curiosidad (clics) pero nula confianza u organización (votos), patrón típico de un bulo (noticia falsa)
UMBRAL_SOSPECHA = 3.0
df_noticias["¿Es_Bulo_Sospechoso?"] = df_noticias["Ratio_Validacion_%"] < UMBRAL_SOSPECHA

# Separar los resultados para el reporte
noticias_validas = df_noticias[df_noticias["¿Es_Bulo_Sospechoso?"] == False]
noticias_falsas_probables = df_noticias[df_noticias["¿Es_Bulo_Sospechoso?"] == True]


# IMPRESIÓN DIRECTA DE RESULTADOS

print("\n--- RESULTADOS DEL ANÁLISIS ---")
print(f"Total de noticias evaluadas: {len(df_noticias)}")
print(f"Noticias orgánicas/válidas : {len(noticias_validas)}")
print(f"Alertas de posibles desinformaciones o bulos: {len(noticias_falsas_probables)}")

print("\n--- NOTICIAS FALSAS / BULOS DETECTADOS EN EL RESULTADO ---")
if not noticias_falsas_probables.empty:
    # Mostramos únicamente las columnas analíticas que comprueban el fraude
    print(noticias_falsas_probables[["Título", "Clics", "Votos", "Ratio_Validacion_%"]])
else:
    print("No se detectaron anomalías críticas en este lote de páginas.")

# Guardar los datos en la tabla CSV final
df_noticias.to_csv("reporte_meneame_limpio.csv", index=False, encoding="utf-8")

Extrayendo datos de: https://www.meneame.net/?page=1
Extrayendo datos de: https://www.meneame.net/?page=2

--- RESULTADOS DEL ANÁLISIS ---
Total de noticias evaluadas: 50
Noticias orgánicas/válidas : 48
Alertas de posibles desinformaciones o bulos: 2

--- NOTICIAS FALSAS / BULOS DETECTADOS EN EL RESULTADO ---
                                               Título  Clics  Votos  \
11  Joan Collins: impactantes retratos del rodaje ...   1756     47   
22  "Atropellado en León un anciano de 45 años a l...  10757    194   

    Ratio_Validacion_%  
11            2.675014  
22            1.803309  
